In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 68 — MCP-Powered IDE Assistant

## Goal
Build an **IDE coding assistant** that uses **structured tools**
to search docs, read code, explain patterns, and suggest fixes —
demonstrating the **Model Context Protocol (MCP)** concept.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **MCP pattern** | Standardized tool protocol for LLM ↔ tools |
| **Code tools** | Read files, search code, explain snippets |
| **Doc retrieval** | Fetch library documentation |
| **IDE integration** | How coding assistants work under the hood |

## What is MCP?
```
Model Context Protocol (MCP) is an open standard by Anthropic that defines
how LLMs communicate with external tools and data sources.

Instead of custom API integrations, MCP provides:
- A standard protocol for tool discovery & invocation
- Server-side tool definitions (resources, tools, prompts)
- Client-side tool calling from any LLM

Think of it as "USB for AI" — plug in any tool, any model.
```

## Architecture
```
User (coding question)
    ↓
LLM Agent
    ↓ (tool calls via MCP-style protocol)
┌─────────────────────────────────────┐
│ Tool Server (simulated MCP)         │
│  ├── read_source_file()             │
│  ├── search_codebase()              │
│  ├── search_documentation()         │
│  ├── explain_code()                 │
│  └── suggest_fix()                  │
└─────────────────────────────────────┘
```

## Stack
- `LangGraph` + `ChatOllama`
- Simulated repo + docs as MCP resources

In [ ]:
import os, warnings, json, re
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from pathlib import Path
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

print("All imports OK")

## Step 1 — Create a Simulated Code Repository

We build a small Python project that the agent can explore.

In [ ]:
import shutil

REPO_DIR = Path("./sim_project")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

files = {
    "app/main.py": '''"""FastAPI application entry point."""\nfrom fastapi import FastAPI, HTTPException\nfrom app.models import User, UserCreate\nfrom app.database import get_db, create_user, get_user_by_email\n\napp = FastAPI(title="User Service", version="1.0")\n\n@app.post("/users", response_model=User)\nasync def create_new_user(user: UserCreate):\n    existing = get_user_by_email(user.email)\n    if existing:\n        raise HTTPException(status_code=400, detail="Email already registered")\n    return create_user(user)\n\n@app.get("/users/{user_id}", response_model=User)\nasync def read_user(user_id: int):\n    user = get_db().get(user_id)\n    if not user:\n        raise HTTPException(status_code=404, detail="User not found")\n    return user\n\n@app.get("/health")\nasync def health_check():\n    return {"status": "ok"}\n''',
    "app/models.py": '''"""Pydantic models for request/response validation."""\nfrom pydantic import BaseModel, EmailStr\nfrom typing import Optional\n\nclass UserCreate(BaseModel):\n    name: str\n    email: str  # TODO: should use EmailStr for validation\n    age: Optional[int] = None\n\nclass User(UserCreate):\n    id: int\n    is_active: bool = True\n''',
    "app/database.py": '''"""Simple in-memory database."""\n\n_db = {}\n_counter = 0\n\ndef get_db():\n    return _db\n\ndef create_user(user_data):\n    global _counter\n    _counter += 1\n    record = {"id": _counter, **user_data.dict(), "is_active": True}\n    _db[_counter] = record\n    return record\n\ndef get_user_by_email(email):\n    for user in _db.values():\n        if user["email"] == email:\n            return user\n    return None\n\ndef delete_user(user_id):\n    # BUG: doesn't check if user exists before deleting\n    del _db[user_id]\n''',
    "app/__init__.py": "",
    "tests/test_main.py": '''"""Tests for main API."""\nimport pytest\n\ndef test_create_user():\n    # TODO: implement\n    pass\n\ndef test_duplicate_email():\n    # TODO: implement\n    pass\n\ndef test_user_not_found():\n    # TODO: implement\n    pass\n''',
    "README.md": "# User Service\n\nA simple FastAPI user management service.\n\n## Setup\n```\npip install fastapi uvicorn\nuvicorn app.main:app --reload\n```\n\n## Endpoints\n- POST /users - Create user\n- GET /users/{id} - Get user\n- GET /health - Health check\n",
}

DOCS = {
    "fastapi": """FastAPI Documentation Summary:\n- FastAPI is a modern Python web framework for building APIs\n- Uses Pydantic models for request/response validation\n- Automatic OpenAPI documentation at /docs\n- Supports async/await\n- HTTPException(status_code, detail) for error responses\n- @app.get/@app.post decorators for routes\n- response_model parameter validates output""",
    "pydantic": """Pydantic Documentation Summary:\n- BaseModel for data validation\n- EmailStr for email validation (requires email-validator)\n- Optional[type] for nullable fields\n- Field() for constraints (min_length, max_length, gt, lt)\n- model.dict() → dictionary (v1), model.model_dump() → dict (v2)\n- validator / field_validator for custom validation""",
    "pytest": """Pytest Documentation Summary:\n- pytest fixtures for test setup/teardown\n- @pytest.fixture for dependency injection\n- pytest.raises(ExceptionType) for exception testing\n- conftest.py for shared fixtures\n- assert statement for test assertions\n- parametrize decorator for test variations""",
}

for path, content in files.items():
    fp = REPO_DIR / path
    fp.parent.mkdir(parents=True, exist_ok=True)
    fp.write_text(content)

print(f"Created simulated repo with {len(files)} files")
print(f"Documentation available for: {list(DOCS.keys())}")

## Step 2 — Define MCP-Style Tools

These tools simulate what an MCP server provides to an IDE assistant.

In [ ]:
@tool
def read_source_file(file_path: str) -> str:
    """Read a source file from the project. Paths are relative to project root (e.g., 'app/main.py')."""
    target = (REPO_DIR / file_path).resolve()
    if not str(target).startswith(str(REPO_DIR.resolve())):
        return "Error: Path traversal blocked."
    if not target.exists():
        return f"File not found: {file_path}"
    content = target.read_text()
    lines = content.split("\n")
    numbered = "\n".join(f"{i+1:3d} | {line}" for i, line in enumerate(lines))
    return f"File: {file_path} ({len(lines)} lines)\n{'─'*50}\n{numbered}"

@tool
def search_codebase(query: str) -> str:
    """Search the entire codebase for a string (function name, variable, pattern). Returns matching lines with file and line number."""
    results = []
    for path_str in files:
        content = (REPO_DIR / path_str).read_text()
        for i, line in enumerate(content.split("\n"), 1):
            if query.lower() in line.lower():
                results.append(f"{path_str}:{i}  {line.strip()}")
    return "\n".join(results) if results else f"No matches for '{query}'."

@tool
def list_project_files() -> str:
    """List all files in the project with their sizes."""
    entries = []
    for path_str in sorted(files.keys()):
        fp = REPO_DIR / path_str
        size = fp.stat().st_size
        entries.append(f"  {path_str} ({size} bytes)")
    return "Project files:\n" + "\n".join(entries)

@tool
def search_documentation(library: str) -> str:
    """Look up documentation for a library. Available: fastapi, pydantic, pytest."""
    lib = library.lower()
    if lib in DOCS:
        return DOCS[lib]
    return f"No docs for '{library}'. Available: {list(DOCS.keys())}"

@tool
def find_issues() -> str:
    """Scan the codebase for common issues: TODOs, BUGs, potential problems."""
    issues = []
    patterns = {
        "TODO": r"#\s*TODO",
        "BUG": r"#\s*BUG",
        "FIXME": r"#\s*FIXME",
        "bare except": r"except\s*:",
        "hardcoded": r"(password|secret|key)\s*=\s*[\"']",
    }
    for path_str, content_str in files.items():
        real_content = (REPO_DIR / path_str).read_text()
        for i, line in enumerate(real_content.split("\n"), 1):
            for label, pattern in patterns.items():
                if re.search(pattern, line, re.IGNORECASE):
                    issues.append(f"[{label}] {path_str}:{i}  {line.strip()}")
    return "\n".join(issues) if issues else "No issues found."

ide_tools = [read_source_file, search_codebase, list_project_files, search_documentation, find_issues]
print(f"Defined {len(ide_tools)} MCP-style tools")

## Step 3 — Create the IDE Agent

In [ ]:
llm = ChatOllama(model="qwen3.5:9b", temperature=0)

IDE_PROMPT = """You are an AI coding assistant integrated into an IDE.

You help developers by:
1. Reading and explaining source code
2. Searching the codebase for functions, variables, or patterns
3. Looking up library documentation
4. Finding bugs, TODOs, and code quality issues
5. Suggesting fixes and improvements

Rules:
- Always read the relevant code before answering
- Reference specific file names and line numbers
- Provide code examples when suggesting fixes
- Check documentation for correct API usage
- Be concise but thorough"""

ide_agent = create_react_agent(
    model=llm,
    tools=ide_tools,
    prompt=IDE_PROMPT,
)

def ask_ide(question: str):
    print(f"\n{'='*70}")
    print(f"DEV: {question}")
    print(f"{'='*70}")
    result = ide_agent.invoke({"messages": [{"role": "user", "content": question}]})
    for msg in result["messages"]:
        role = msg.__class__.__name__
        if role == "AIMessage" and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"\n🔧 TOOL: {tc['name']}({str(tc['args'])[:100]})")
        elif role == "ToolMessage":
            print(f"📋 RESULT: {msg.content[:400]}")
        elif role == "AIMessage" and msg.content:
            print(f"\n🤖 ASSISTANT:\n{msg.content}")
    return result

print("IDE assistant ready!")

In [ ]:
# Task 1: Code exploration
ask_ide("What does this project do? Give me an overview of the architecture.")

In [ ]:
# Task 2: Bug finding
ask_ide("Scan the codebase for bugs and issues. What should I fix first?")

In [ ]:
# Task 3: Feature help with docs
ask_ide("I want to add email validation to the UserCreate model. How should I do it? Check the pydantic docs.")

## Step 4 — How This Relates to MCP

Our tools follow the same **pattern** as MCP, even though we
used LangChain's `@tool` decorator instead of the MCP protocol.

In [ ]:
# Conceptual MCP server definition
mcp_server_config = {
    "name": "ide-assistant",
    "version": "1.0",
    "description": "IDE coding assistant MCP server",
    "tools": [
        {
            "name": "read_source_file",
            "description": "Read a source file from the project",
            "inputSchema": {
                "type": "object",
                "properties": {"file_path": {"type": "string", "description": "Path relative to project root"}},
                "required": ["file_path"],
            },
        },
        {
            "name": "search_codebase",
            "description": "Search all files for a pattern",
            "inputSchema": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
            },
        },
    ],
    "resources": [
        {"uri": "docs://fastapi", "name": "FastAPI Documentation"},
        {"uri": "docs://pydantic", "name": "Pydantic Documentation"},
    ],
}

print("MCP Server Configuration:")
print(json.dumps(mcp_server_config, indent=2))

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **MCP tools** | Structured functions with schemas the LLM can call |
| **MCP resources** | Read-only data sources (docs, code files) |
| **Code reading** | File content with line numbers for precise references |
| **Issue detection** | Regex-based scanning for TODOs, BUGs, patterns |
| **Doc retrieval** | Library documentation as a tool resource |

## MCP vs. Direct Tool Calling
```
Without MCP:                       With MCP:
┌──────┐  Custom API  ┌──────┐    ┌──────┐  Standard  ┌──────────┐
│ LLM  │───────────→│ Tool │    │ LLM  │──Protocol──│MCP Server│
└──────┘              └──────┘    └──────┘            │  ├─ Tool │
                                                       │  ├─ Tool │
Each LLM needs custom           Any LLM can connect  │  └─ Tool │
integration per tool.            to any MCP server.   └──────────┘
```

## Real MCP Servers for IDE
- **Filesystem MCP**: Read/write files in workspace
- **Git MCP**: Commit history, diffs, branches
- **Database MCP**: Schema, queries
- **Browser MCP**: Fetch web content, docs
- **GitHub MCP**: Issues, PRs, reviews